# Notebook 01 — CNN Baseline + MobileNetV2 Student

**NeuroDriver CNN ADAS Colombia** — Phase 1 milestone.

Designed to run on Google Colab (mount Drive / clone repo) but portable to local Windows/Linux.
Full MobileNetV2 training, Colombian fine-tuning, and Knowledge Distillation are **not** executed
in this notebook — see Sections 10-12 for what is prepared vs. pending.


## 1. Environment / GPU check

In [ ]:
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "00_check_environment.py")])


## 2. Deterministic seeds

In [ ]:
from neurodriver_cnn.utils.seed import set_global_seed
from neurodriver_cnn.config import load_dataset_config, load_training_config

SEED = load_dataset_config(PROJECT_ROOT)["seed"]
set_global_seed(SEED)
print(f"Seed set to {SEED}")


## 3. Configurable project / data root

No personal/hardcoded Google Drive paths — override `DATA_ROOT` below only if your data lives outside `PROJECT_ROOT/data`.

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data"
MANIFEST_PATH = DATA_ROOT / "processed" / "manifests" / "experiment_manifest.csv"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")


## 4. Load Common Manifest + label validation

In [ ]:
import pandas as pd
from neurodriver_cnn.data.manifest import validate_manifest_invariants

if MANIFEST_PATH.exists():
    manifest_df = pd.read_csv(MANIFEST_PATH)
    violations = validate_manifest_invariants(manifest_df)
    print(f"Loaded {len(manifest_df)} rows.")
    if violations:
        print("VIOLATIONS:")
        for v in violations:
            print(f" - {v}")
    else:
        print("[OK] Manifest passed invariant validation.")
else:
    manifest_df = None
    print("PENDING: experiment_manifest.csv not found.")
    print("Run scripts/00-04 after placing BDD100K data (see docs/bdd100k_setup.md).")


## 5. tf.data pipeline (TRAIN/VAL/TEST)

In [ ]:
import tensorflow as tf

IMAGE_SIZE = tuple(load_dataset_config(PROJECT_ROOT)["image_size"])
BATCH_SIZE = load_training_config(PROJECT_ROOT)["batch_size"]
CLASS_NAMES = ["CLEAR", "VEHICLE", "PEDESTRIAN", "MIXED"]
CLASS_TO_INDEX = {n: i for i, n in enumerate(CLASS_NAMES)}


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return image, label


def make_dataset(df: pd.DataFrame, split: str, shuffle: bool) -> tf.data.Dataset:
    split_df = df[df["split"] == split]
    paths = split_df["image_path"].tolist()
    labels = [CLASS_TO_INDEX[l] for l in split_df["label"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=max(1, len(paths)), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


if manifest_df is not None:
    train_ds = make_dataset(manifest_df, "TRAIN", shuffle=True)
    val_ds = make_dataset(manifest_df, "VALIDATION", shuffle=False)
    test_ds = make_dataset(manifest_df, "TEST", shuffle=False)
    print("[OK] tf.data pipelines built.")
else:
    train_ds = val_ds = test_ds = None
    print("PENDING: cannot build tf.data pipelines without a manifest.")


## 6. Train-only augmentation

Applied **only** to `train_ds`, after the split (no leakage): horizontal flip, modest
brightness/contrast, modest zoom. **Never** vertical flip or 90/180-degree rotation — the current
4-class labels carry no left/right semantics that a horizontal flip would violate, but a vertical
flip or road rotation would produce physically nonsensical driving scenes.


In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomBrightness(0.1),
        tf.keras.layers.RandomContrast(0.1),
        tf.keras.layers.RandomZoom(0.1),
    ],
    name="train_augmentation",
)


def augment(image, label):
    return augmentation(image, training=True), label


if train_ds is not None:
    train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    print("[OK] Augmentation attached to train_ds only.")


## 7. Simple CNN baseline

In [ ]:
from neurodriver_cnn.models.baseline import build_baseline_cnn, compile_baseline, build_baseline_callbacks

training_config = load_training_config(PROJECT_ROOT)

baseline_model = build_baseline_cnn(
    input_shape=IMAGE_SIZE + (3,), dropout=training_config["baseline"]["dropout"]
)
compile_baseline(baseline_model, learning_rate=training_config["baseline"]["learning_rate"])
baseline_model.summary()


## 8. Callbacks + baseline evaluation

In [ ]:
callbacks = build_baseline_callbacks(
    patience_es=training_config["callbacks"]["early_stopping_patience"],
    patience_lr=training_config["callbacks"]["reduce_lr_patience"],
)

BASELINE_STATUS = "PENDING"
if train_ds is not None and val_ds is not None:
    # A short run is acceptable for honest preliminary metrics; NOT a long
    # production training run. Increase epochs deliberately, not automatically.
    history = baseline_model.fit(
        train_ds, validation_data=val_ds, epochs=training_config["baseline"]["epochs"], callbacks=callbacks
    )
    BASELINE_STATUS = "DONE (preliminary short run)"
else:
    print("PENDING: baseline training skipped, no data available yet.")

print(f"Baseline status: {BASELINE_STATUS}")


In [ ]:
from neurodriver_cnn.evaluation.metrics import classification_metrics, confusion_matrix, evaluate_motorcycle_subset
import numpy as np

if test_ds is not None and BASELINE_STATUS.startswith("DONE"):
    y_true = np.concatenate([y.numpy() for _, y in test_ds])
    y_pred = np.argmax(baseline_model.predict(test_ds), axis=1)
    print(classification_metrics(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))
else:
    print("PENDING: baseline evaluation requires a trained baseline and TEST data.")


## 9. MobileNetV2 Student construction

In [ ]:
from neurodriver_cnn.models.mobilenetv2 import build_mobilenetv2_student, compile_student, count_parameters

student_model, student_logits_model, student_base_model = build_mobilenetv2_student(
    input_shape=IMAGE_SIZE + (3,), dropout=training_config["mobilenetv2"]["dropout"], freeze_backbone=True
)
compile_student(student_model, learning_rate=training_config["mobilenetv2"]["head_learning_rate"])
student_model.summary()
print(count_parameters(student_model))


## 10. Forward-pass smoke test

In [ ]:
MOBILENET_SMOKE_TEST_STATUS = "PENDING"
if train_ds is not None:
    for images, labels in train_ds.take(1):
        probs = student_model.predict(images, verbose=0)
        logits = student_logits_model.predict(images, verbose=0)
        assert probs.shape == (images.shape[0], 4), probs.shape
        assert not np.isnan(probs).any(), "NaN in student predictions"
        MOBILENET_SMOKE_TEST_STATUS = "DONE"
        print(f"[OK] probs shape={probs.shape}, logits shape={logits.shape}, no NaNs.")
else:
    print("PENDING: forward-pass smoke test requires real data (a real batch), not synthetic tensors.")

print(f"MobileNetV2 smoke-test status: {MOBILENET_SMOKE_TEST_STATUS}")


## 11. Future fine-tuning cells (prepared, not executed)

Fine-tuning procedure once BDD supervised training on the head is solid:

1. train the head with the backbone frozen (Section 9-10, done above);
2. `unfreeze_for_fine_tuning(student_base_model, unfreeze_from_layer=...)`;
3. recompile with a low learning rate (`training_config["mobilenetv2"]["fine_tune_learning_rate"]`, ~1e-5);
4. fine-tune carefully; BatchNormalization layers are kept frozen (inference mode) even when
   unfrozen, since small BDD fine-tuning batches are not representative enough to safely update
   BatchNorm running statistics;
5. keep BDD source-domain fine-tuning and later Colombian target-domain fine-tuning as **separate,
   comparable experiments** (see `docs/colombian_domain_strategy.md`).


In [ ]:
from neurodriver_cnn.models.mobilenetv2 import unfreeze_for_fine_tuning

FINE_TUNING_STATUS = "PENDING (not executed in Phase 1)"
# Example of what Phase 2 will run — intentionally not executed here:
# unfreeze_for_fine_tuning(student_base_model, unfreeze_from_layer=training_config["mobilenetv2"]["fine_tune_unfreeze_from_layer"])
# compile_student(student_model, learning_rate=training_config["mobilenetv2"]["fine_tune_learning_rate"])
# student_model.fit(train_ds, validation_data=val_ds, epochs=training_config["mobilenetv2"]["fine_tune_epochs"], callbacks=callbacks)
print(FINE_TUNING_STATUS)


## 12. Knowledge-Distillation readiness

- `student_logits_model` above exposes pre-Softmax logits directly, as required for a future
  distillation loss (`docs/teacher_student_contract.md`).
- No Teacher model, Teacher logits, or KD loss exists yet — `configs/training_config.json`
  keeps `distillation.enabled = false`.
- See `src/neurodriver_cnn/distillation/README.md` for the planned `TeacherAdapter` /
  distillation-loss / `Distiller` components.
- **No fake KD is performed here.**


## 13. Status summary

| Component | Status |
|---|---|
| Environment/seed setup | DONE |
| Manifest loading + validation | DONE if `experiment_manifest.csv` exists, else PENDING |
| tf.data pipeline + augmentation | DONE if data available, else PENDING |
| Baseline CNN training | see `BASELINE_STATUS` above |
| Baseline evaluation | PENDING unless baseline trained and TEST data available |
| MobileNetV2 Student construction | DONE |
| MobileNetV2 forward-pass smoke test | see `MOBILENET_SMOKE_TEST_STATUS` above |
| MobileNetV2 full training | NOT required for Phase 1 |
| Fine-tuning | PENDING (Phase 2+) |
| Knowledge Distillation | PENDING (Phase 2+, requires a NeuroDriver Teacher) |
